<center><h2><b>Natural Language Processing (NLP)</b></h2>
<h3><b>Notebook 2: Arabic Question Answering</b></h3>
<h4>Student: Hala Nazzal</h4><hr></center>

**Approaches:**
- **Part 1:** Retrieval-based QA using RNN / LSTM / GRU / Bidirectional GRU (category classifier + cosine similarity)
- **Part 2:** Seq2Seq Model with Various Word Embedding Methods (Traditional / Word2Vec / AraBERT)
- **Part 3:** Fine-tuned Transformer Models — AraGPT2 (GPT), mT5-small (T5), Qwen2.5-0.5B (local, no API)


In [1]:
!pip install transformers torch tensorflow scikit-learn pandas numpy -q


In [2]:
import re
import nltk
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModel
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (Dense, Dropout, SimpleRNN, LSTM,
                                     GRU, Bidirectional, Input)
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords', quiet=True)
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")


Device: cuda


### Load Preprocessed Data
---

In [6]:
df = pd.read_csv('/content/AAFAQ_Preprocessed.csv', encoding='utf-8-sig')
print(f"Shape: {df.shape}")
df.head()

Shape: (4522, 5)


,Question,Answer,Category,cleaned_question,stemmed_question_text
0,أيهما أفضل الدراسة في السابق أم في الوقت الحالي؟,الدراسة في الوقت الحالي تعتبر أفضل بسبب توفر ا...,التعليم,ايهما افضل الدراسة السابق ام الوقت الحالي؟,ايه فضل درس سبق ام وقت الحالي
1,أليس القطن عماد الثروة في مصر؟,القطن يعتبر من أهم المنتجات الزراعية في مصر، و...,الاقتصاد والعمل,اليس القطن عماد الثروة مصر؟,الس قطن عمد ثرة مصر
2,أتصعد الشمس من الشرق؟,الشمس تصعد من الشرق.,التعليم,اتصعد الشمس الشرق؟,صعد شمس شرق
3,أتعرف البكتيريا بأنها كائنات حية دقيقة؟,البكتيريا تُعرف بأنها كائنات حية دقيقة.,التعليم,اتعرف البكتيريا بانها كائنات حية دقيقة؟,عرف كتر بان كئن حية دقق
4,أيتكون الهواء أساساً من النيتروجين؟,الهواء يتكون أساساً من النيتروجين.,التعليم,ايتكون الهواء اساسا النيتروجين؟,ايت هوء سسا ترج


### Data Split
---

In [7]:
label_encoder = LabelEncoder()
df['label'] = label_encoder.fit_transform(df['Category'])

X = df['stemmed_question_text'].astype(str)
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Classes: {len(label_encoder.classes_)}")


Train: 3617 | Test: 905
Classes: 16


---
## Word Representations for Sequential Models
---

### 1- Traditional (Keras Tokenizer + Learned Embedding Layer):

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer as KerasTokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding

MAX_VOCAB  = 20000
MAX_LEN_T  = 50

# Fit Keras tokenizer on training text
keras_tok = KerasTokenizer(num_words=MAX_VOCAB, oov_token='<OOV>')
keras_tok.fit_on_texts(X_train.tolist())

# Convert text to padded sequences
X_train_trad = pad_sequences(keras_tok.texts_to_sequences(X_train.tolist()), maxlen=MAX_LEN_T, padding='post')
X_test_trad  = pad_sequences(keras_tok.texts_to_sequences(X_test.tolist()),  maxlen=MAX_LEN_T, padding='post')
print(f'Traditional sequences shape: {X_train_trad.shape}')

Traditional sequences shape: (3617, 50)


In [9]:
num_classes = len(label_encoder.classes_)

# RNN — Traditional
rnn_trad = Sequential([
    Embedding(MAX_VOCAB, 128, input_length=MAX_LEN_T),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
rnn_trad.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_trad.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_rnn_trad = rnn_trad.evaluate(X_test_trad, y_test)
print(f'RNN  (Traditional): {acc_rnn_trad:.4f}')

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 16s 88ms/step - accuracy: 0.1263 - loss: 2.5563 - val_accuracy: 0.1215 - val_loss: 2.5216
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.1576 - loss: 2.4337 - val_accuracy: 0.2431 - val_loss: 2.3772
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.2120 - loss: 2.2967 - val_accuracy: 0.2348 - val_loss: 2.2689
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2535 - loss: 2.1522 - val_accuracy: 0.2072 - val_loss: 2.2808
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.2618 - loss: 2.0699 - val_accuracy: 0.2762 - val_loss: 2.1593
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.3327 - loss: 1.9014 - val_accuracy: 0.2901 - val_loss: 2.2691
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.3975 - loss: 1.7243 - val_accuracy: 0.3287 - val_loss: 2.0153
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.3115 - loss: 2.1022 - val_accuracy: 0.2790 - 

In [10]:
# LSTM — Traditional
lstm_trad = Sequential([
    Embedding(MAX_VOCAB, 128, input_length=MAX_LEN_T),
    LSTM(128),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
lstm_trad.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_trad.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_lstm_trad = lstm_trad.evaluate(X_test_trad, y_test)
print(f'LSTM (Traditional): {acc_lstm_trad:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 12s 17ms/step - accuracy: 0.1361 - loss: 2.5292 - val_accuracy: 0.1298 - val_loss: 2.5176
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1515 - loss: 2.4426 - val_accuracy: 0.1519 - val_loss: 2.4988
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1548 - loss: 2.4400 - val_accuracy: 0.1050 - val_loss: 2.5064
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1567 - loss: 2.4390 - val_accuracy: 0.1519 - val_loss: 2.4982
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1502 - loss: 2.4374 - val_accuracy: 0.1519 - val_loss: 2.4915
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.1493 - loss: 2.4374 - val_accuracy: 0.1519 - val_loss: 2.4922
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.1564 - loss: 2.4372 - val_accuracy: 0.1519 - val_loss: 2.5107
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1564 - loss: 2.4375 - val_accuracy: 0.1519 - 

In [11]:
# GRU — Traditional
gru_trad = Sequential([
    Embedding(MAX_VOCAB, 128, input_length=MAX_LEN_T),
    GRU(128),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
gru_trad.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_trad.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_gru_trad = gru_trad.evaluate(X_test_trad, y_test)
print(f'GRU  (Traditional): {acc_gru_trad:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.1355 - loss: 2.5018 - val_accuracy: 0.1519 - val_loss: 2.5128
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.1564 - loss: 2.4451 - val_accuracy: 0.1519 - val_loss: 2.4963
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1496 - loss: 2.4387 - val_accuracy: 0.1519 - val_loss: 2.4983
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1564 - loss: 2.4400 - val_accuracy: 0.1519 - val_loss: 2.4986
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1490 - loss: 2.4383 - val_accuracy: 0.1519 - val_loss: 2.4955
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1564 - loss: 2.4360 - val_accuracy: 0.1519 - val_loss: 2.4986
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1564 - loss: 2.4364 - val_accuracy: 0.1519 - val_loss: 2.4980
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.1515 - loss: 2.4386 - val_accuracy: 0.1519 - val_loss

In [12]:
# Bidirectional GRU — Traditional
bigru_trad = Sequential([
    Embedding(MAX_VOCAB, 128, input_length=MAX_LEN_T),
    Bidirectional(GRU(128)),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
bigru_trad.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
bigru_trad.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_bigru_trad = bigru_trad.evaluate(X_test_trad, y_test)
print(f'BiGRU (Traditional): {acc_bigru_trad:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.1518 - loss: 2.4982 - val_accuracy: 0.2099 - val_loss: 2.3832
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.3693 - loss: 1.9565 - val_accuracy: 0.4365 - val_loss: 1.7233
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6147 - loss: 1.2331 - val_accuracy: 0.5635 - val_loss: 1.4075
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.7518 - loss: 0.8243 - val_accuracy: 0.5967 - val_loss: 1.3364
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8292 - loss: 0.5625 - val_accuracy: 0.5994 - val_loss: 1.3310
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.8750 - loss: 0.4114 - val_accuracy: 0.6077 - val_loss: 1.4688
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9014 - loss: 0.3147 - val_accuracy: 0.6077 - val_loss: 1.5966
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.9257 - loss: 0.2483 - val_accuracy: 0.6354 - v

### 2- Independent (Word2Vec — context-independent static embeddings):

In [13]:
!pip install gensim -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 71.6 MB/s eta 0:00:00


In [14]:
from gensim.models import Word2Vec

# Train Word2Vec on training corpus
sentences = [text.split() for text in X_train.tolist()]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Build embedding matrix from Word2Vec
vocab_size_w2v = len(keras_tok.word_index) + 1
EMB_DIM        = 100
embedding_matrix = np.zeros((vocab_size_w2v, EMB_DIM))

for word, idx in keras_tok.word_index.items():
    if word in w2v_model.wv:
        embedding_matrix[idx] = w2v_model.wv[word]

print(f'Embedding matrix shape: {embedding_matrix.shape}')

Embedding matrix shape: (2258, 100)


In [15]:
# RNN — Independent (Word2Vec, frozen)
rnn_indep = Sequential([
    Embedding(vocab_size_w2v, EMB_DIM, weights=[embedding_matrix], trainable=False, input_length=MAX_LEN_T),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
rnn_indep.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_indep.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_rnn_indep = rnn_indep.evaluate(X_test_trad, y_test)
print(f'RNN  (Independent): {acc_rnn_indep:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 19s 179ms/step - accuracy: 0.1324 - loss: 2.5108 - val_accuracy: 0.1271 - val_loss: 2.5228
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.1767 - loss: 2.4233 - val_accuracy: 0.1768 - val_loss: 2.4917
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1714 - loss: 2.4176 - val_accuracy: 0.1547 - val_loss: 2.4752
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.1773 - loss: 2.4002 - val_accuracy: 0.1685 - val_loss: 2.4773
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.1923 - loss: 2.3916 - val_accuracy: 0.1713 - val_loss: 2.4744
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.1877 - loss: 2.3994 - val_accuracy: 0.1381 - val_loss: 2.4983
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.1862 - loss: 2.3899 - val_accuracy: 0.1602 - val_loss: 2.4857
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 35ms/step - accuracy: 0.1963 - loss: 2.3682 - val_accuracy: 0.1934 -

In [16]:
# LSTM — Independent
lstm_indep = Sequential([
    Embedding(vocab_size_w2v, EMB_DIM, weights=[embedding_matrix], trainable=False, input_length=MAX_LEN_T),
    LSTM(128),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
lstm_indep.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_indep.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_lstm_indep = lstm_indep.evaluate(X_test_trad, y_test)
print(f'LSTM (Independent): {acc_lstm_indep:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 4s 36ms/step - accuracy: 0.1296 - loss: 2.5392 - val_accuracy: 0.1519 - val_loss: 2.5013
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.1410 - loss: 2.4433 - val_accuracy: 0.1519 - val_loss: 2.5037
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.1564 - loss: 2.4410 - val_accuracy: 0.1519 - val_loss: 2.4974
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 26ms/step - accuracy: 0.1515 - loss: 2.4392 - val_accuracy: 0.1519 - val_loss: 2.4971
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1536 - loss: 2.4371 - val_accuracy: 0.1519 - val_loss: 2.4935
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1469 - loss: 2.4398 - val_accuracy: 0.1519 - val_loss: 2.4933
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - accuracy: 0.1508 - loss: 2.4384 - val_accuracy: 0.1519 - val_loss: 2.4920
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 19ms/step - accuracy: 0.1462 - loss: 2.4376 - val_accuracy: 0.1519 - v

In [17]:
# GRU — Independent
gru_indep = Sequential([
    Embedding(vocab_size_w2v, EMB_DIM, weights=[embedding_matrix], trainable=False, input_length=MAX_LEN_T),
    GRU(128),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
gru_indep.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_indep.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_gru_indep = gru_indep.evaluate(X_test_trad, y_test)
print(f'GRU  (Independent): {acc_gru_indep:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 33ms/step - accuracy: 0.1521 - loss: 2.6033 - val_accuracy: 0.1519 - val_loss: 2.5202
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 29ms/step - accuracy: 0.1564 - loss: 2.4611 - val_accuracy: 0.1519 - val_loss: 2.5163
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 34ms/step - accuracy: 0.1564 - loss: 2.4429 - val_accuracy: 0.1519 - val_loss: 2.5005
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.1564 - loss: 2.4402 - val_accuracy: 0.1519 - val_loss: 2.4975
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.1564 - loss: 2.4358 - val_accuracy: 0.1519 - val_loss: 2.4888
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.1564 - loss: 2.4355 - val_accuracy: 0.1519 - val_loss: 2.4964
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 26ms/step - accuracy: 0.1564 - loss: 2.4359 - val_accuracy: 0.1519 - val_loss: 2.4890
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 33ms/step - accuracy: 0.1564 - loss: 2.4373 - val_accuracy: 0.1519 - v

In [18]:
# Bidirectional GRU — Independent
bigru_indep = Sequential([
    Embedding(vocab_size_w2v, EMB_DIM, weights=[embedding_matrix], trainable=False, input_length=MAX_LEN_T),
    Bidirectional(GRU(128)),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])
bigru_indep.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
bigru_indep.fit(X_train_trad, y_train, epochs=10, batch_size=64, validation_split=0.1)
_, acc_bigru_indep = bigru_indep.evaluate(X_test_trad, y_test)
print(f'BiGRU (Independent): {acc_bigru_indep:.4f}')

Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 5s 38ms/step - accuracy: 0.1487 - loss: 2.5209 - val_accuracy: 0.1381 - val_loss: 2.5045
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.1911 - loss: 2.3952 - val_accuracy: 0.1519 - val_loss: 2.4770
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 47ms/step - accuracy: 0.2077 - loss: 2.3546 - val_accuracy: 0.1657 - val_loss: 2.4748
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.2114 - loss: 2.3333 - val_accuracy: 0.1519 - val_loss: 2.4651
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.2200 - loss: 2.3129 - val_accuracy: 0.1685 - val_loss: 2.4620
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2307 - loss: 2.3054 - val_accuracy: 0.1796 - val_loss: 2.4561
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 42ms/step - accuracy: 0.2332 - loss: 2.2855 - val_accuracy: 0.1906 - val_loss: 2.4257
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step - accuracy: 0.2458 - loss: 2.2725 - val_accuracy: 0.1796 - v

### 3- Dependent (AraBERT — context-aware embeddings):

### AraBERT Embeddings
---

#### Flat Embeddings (for traditional models & cosine similarity):

In [54]:
tokenizer_bert = AutoTokenizer.from_pretrained('aubmindlab/bert-base-arabertv02')
model_bert     = AutoModel.from_pretrained('aubmindlab/bert-base-arabertv02')
model_bert.eval()
model_bert.to(DEVICE)

# Get BERT embeddings for traditional models (mean pooling across all tokens)
def get_bert_embeddings(texts, batch_size=32):
    # If 'texts' is already a list, .tolist() is not needed and causes an error.
    # Handle both Series and list inputs gracefully.
    if isinstance(texts, pd.Series):
        input_texts = texts.tolist()
    elif isinstance(texts, list):
        input_texts = texts
    else:
        raise TypeError("Input 'texts' must be a Pandas Series or a list of strings.")

    inputs = tokenizer_bert(
        input_texts,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=128
    ).to(DEVICE) # Move inputs to the DEVICE
    with torch.no_grad():
        outputs = model_bert(**inputs)
    # Mean pooling across all token positions
    embeddings = outputs.last_hidden_state.mean(dim=1)
    return embeddings.cpu().numpy()

X_train_flat = get_bert_embeddings(X_train)
X_test_flat  = get_bert_embeddings(X_test)
print(f"Embeddings shape: {X_train_flat.shape}")

# Alias used by retrieval function
get_flat_embeddings = get_bert_embeddings
print("AraBERT embeddings ready ")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: aubmindlab/bert-base-arabertv02
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (3617, 768)
AraBERT embeddings ready 


In [55]:
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
weights_dict  = dict(enumerate(class_weights))
print("Class weights computed")

Class weights computed


#### Sequence Embeddings (for RNN/LSTM/GRU — shape: n, MAX_LEN, 768):

In [56]:
MAX_LEN = 128

# Get BERT sequence embeddings for sequential models (full token sequence)
def get_bert_sequence_embeddings(texts, batch_size=32):
    inputs = tokenizer_bert(
        texts.tolist(),
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=MAX_LEN
    ).to(DEVICE) # Move inputs to the DEVICE
    with torch.no_grad():
        outputs = model_bert(**inputs)
    # Return full token sequence shape: (batch, seq_len, 768)
    return outputs.last_hidden_state.cpu().numpy()

import os
if os.path.exists('/content/X_train_seq.npy') and os.path.exists('/content/X_test_seq.npy'):
    X_train_seq = np.load('/content/X_train_seq.npy')
    X_test_seq  = np.load('/content/X_test_seq.npy')
    print(f"Loaded from cache: {X_train_seq.shape}")
else:
    X_train_seq = get_bert_sequence_embeddings(X_train)
    X_test_seq  = get_bert_sequence_embeddings(X_test)
    np.save('/content/X_train_seq.npy', X_train_seq)
    np.save('/content/X_test_seq.npy',  X_test_seq)
    print(f"Computed: {X_train_seq.shape}")

Loaded from cache: (3617, 20, 768)


---
# Part 1: Retrieval-Based QA
## Sequential Models — Category Classifier

#### RNN:

In [57]:
num_classes = len(label_encoder.classes_)

rnn_model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    SimpleRNN(128, return_sequences=True),
    Dropout(0.3),
    SimpleRNN(64),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

rnn_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

Model: "sequential_19"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn_8 (SimpleRNN)        │ (None, 20, 128)        │       114,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_9 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_38 (Dense)                │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_39 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 132,368 (517.06 KB)

 Trainable params: 132,368 (517.06 KB)

 Non-trainable params: 0 (0.00 B)

In [58]:
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
rnn_model.fit(X_train_seq, y_train, epochs=15, batch_size=64,
              validation_split=0.1, callbacks=[es])


Epoch 1/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 19s 211ms/step - accuracy: 0.1361 - loss: 2.5503 - val_accuracy: 0.1823 - val_loss: 2.3957
Epoch 2/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 30ms/step - accuracy: 0.2141 - loss: 2.3437 - val_accuracy: 0.3094 - val_loss: 2.2253
Epoch 3/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.3014 - loss: 2.1391 - val_accuracy: 0.3370 - val_loss: 2.1630
Epoch 4/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - accuracy: 0.3567 - loss: 2.0140 - val_accuracy: 0.3453 - val_loss: 1.9925
Epoch 5/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.4071 - loss: 1.8657 - val_accuracy: 0.3785 - val_loss: 1.9767
Epoch 6/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - accuracy: 0.4525 - loss: 1.7533 - val_accuracy: 0.4309 - val_loss: 1.8896
Epoch 7/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4790 - loss: 1.6231 - val_accuracy: 0.4061 - val_loss: 1.9522
Epoch 8/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.5146 - loss: 1.5307 - val_accuracy: 0.4779 -

In [59]:
loss, acc = rnn_model.evaluate(X_test_seq, y_test)
print(f"RNN Test Accuracy: {acc:.4f}")


29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.4409 - loss: 1.8080
RNN Test Accuracy: 0.4409


#### LSTM:

In [60]:
lstm_model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    LSTM(128),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

lstm_model.compile(loss='sparse_categorical_crossentropy',
                   optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_5 (LSTM)                   │ (None, 128)            │       459,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_40 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_41 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 468,560 (1.79 MB)

 Trainable params: 468,560 (1.79 MB)

 Non-trainable params: 0 (0.00 B)

In [61]:
lstm_model.fit(X_train_seq, y_train, epochs=15, batch_size=64,
               validation_split=0.1, callbacks=[es])

loss, acc = lstm_model.evaluate(X_test_seq, y_test)
print(f"LSTM Test Accuracy: {acc:.4f}")


Epoch 1/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step - accuracy: 0.2175 - loss: 2.3850 - val_accuracy: 0.2597 - val_loss: 2.2311
Epoch 2/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3576 - loss: 2.0195 - val_accuracy: 0.3867 - val_loss: 1.9599
Epoch 3/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4575 - loss: 1.7563 - val_accuracy: 0.4171 - val_loss: 1.8014
Epoch 4/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5293 - loss: 1.5311 - val_accuracy: 0.4669 - val_loss: 1.6750
Epoch 5/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.5730 - loss: 1.3607 - val_accuracy: 0.4365 - val_loss: 1.6947
Epoch 6/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6089 - loss: 1.2450 - val_accuracy: 0.4724 - val_loss: 1.5909
Epoch 7/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.6661 - loss: 1.0511 - val_accuracy: 0.5359 - val_loss: 1.5287
Epoch 8/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.7143 - loss: 0.9230 - val_accuracy: 0.4862 - val

#### GRU:

In [62]:
gru_model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    GRU(128),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

gru_model.compile(loss='sparse_categorical_crossentropy',
                  optimizer='adam', metrics=['accuracy'])
gru_model.summary()

Model: "sequential_21"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru_10 (GRU)                    │ (None, 128)            │       344,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_42 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_43 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 354,128 (1.35 MB)

 Trainable params: 354,128 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

In [63]:
gru_model.fit(X_train_seq, y_train, epochs=15, batch_size=64,
              validation_split=0.1, callbacks=[es])

loss, acc = gru_model.evaluate(X_test_seq, y_test)
print(f"GRU Test Accuracy: {acc:.4f}")


Epoch 1/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 2s 15ms/step - accuracy: 0.2393 - loss: 2.3471 - val_accuracy: 0.3729 - val_loss: 2.0772
Epoch 2/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.4055 - loss: 1.8945 - val_accuracy: 0.4365 - val_loss: 1.8591
Epoch 3/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.5014 - loss: 1.6016 - val_accuracy: 0.4669 - val_loss: 1.6678
29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3613 - loss: 2.0565
GRU Test Accuracy: 0.3613


#### Bidirectional GRU:

In [64]:
bigru_model = Sequential([
    Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
    Bidirectional(GRU(128, return_sequences=False)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

bigru_model.compile(loss='sparse_categorical_crossentropy',
                    optimizer='adam', metrics=['accuracy'])
bigru_model.summary()

Model: "sequential_22"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional_5 (Bidirectional) │ (None, 256)            │       689,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_19 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_44 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_45 (Dense)                │ (None, 16)             │         1,040 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 707,152 (2.70 MB)

 Trainable params: 707,152 (2.70 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
bigru_model.fit(X_train_seq, y_train, epochs=15, batch_size=64,
                validation_split=0.1, callbacks=[es])

loss, acc = bigru_model.evaluate(X_test_seq, y_test)
print(f"BiGRU Test Accuracy: {acc:.4f}")


Epoch 1/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 3s 19ms/step - accuracy: 0.2464 - loss: 2.3027 - val_accuracy: 0.3757 - val_loss: 1.9832
Epoch 2/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.4381 - loss: 1.7752 - val_accuracy: 0.4669 - val_loss: 1.7262
Epoch 3/15
51/51 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.5410 - loss: 1.4316 - val_accuracy: 0.5138 - val_loss: 1.5315
29/29 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3989 - loss: 1.9166
BiGRU Test Accuracy: 0.3989


#### Select Best Sequential Model:

In [66]:
# Compare ALL models across ALL 3 representations
results_all = {
    'RNN   (Traditional)' : acc_rnn_trad,
    'LSTM  (Traditional)' : acc_lstm_trad,
    'GRU   (Traditional)' : acc_gru_trad,
    'BiGRU (Traditional)' : acc_bigru_trad,
    'RNN   (Independent)' : acc_rnn_indep,
    'LSTM  (Independent)' : acc_lstm_indep,
    'GRU   (Independent)' : acc_gru_indep,
    'BiGRU (Independent)' : acc_bigru_indep,
    'RNN   (Dependent)'   : rnn_model.evaluate(X_test_seq,   y_test, verbose=0)[1],
    'LSTM  (Dependent)'   : lstm_model.evaluate(X_test_seq,  y_test, verbose=0)[1],
    'GRU   (Dependent)'   : gru_model.evaluate(X_test_seq,   y_test, verbose=0)[1],
    'BiGRU (Dependent)'   : bigru_model.evaluate(X_test_seq, y_test, verbose=0)[1],
}

best_name  = max(results_all, key=results_all.get)
print('Sequential model comparison across all representations:')
for k, v in results_all.items():
    print(f'  {k}: {v:.4f}')
print(f'\nBest overall: {best_name} — {results_all[best_name]:.4f}')

Sequential model comparison across all representations:
  RNN   (Traditional): 0.3547
  LSTM  (Traditional): 0.1558
  GRU   (Traditional): 0.1558
  BiGRU (Traditional): 0.6022
  RNN   (Independent): 0.2276
  LSTM  (Independent): 0.1558
  GRU   (Independent): 0.1558
  BiGRU (Independent): 0.2564
  RNN   (Dependent): 0.4409
  LSTM  (Dependent): 0.5514
  GRU   (Dependent): 0.3613
  BiGRU (Dependent): 0.3989

Best overall: BiGRU (Traditional) — 0.6022


### Build All-Embeddings Index (for retrieval)
---

In [67]:
# Compute AraBERT flat embeddings for entire dataset (used in cosine similarity at inference)
all_texts = df['stemmed_question_text'].astype(str)
all_flat_emb = get_flat_embeddings(all_texts)
print(f"Full dataset embeddings shape: {all_flat_emb.shape}")


Full dataset embeddings shape: (4522, 768)


#### Retrieval-Based QA Function:

In [68]:
!pip install pyarabic
import re
from nltk.corpus import stopwords

arabic_stopwords = set(stopwords.words('arabic'))
import pyarabic.araby as araby
from nltk.stem.isri import ISRIStemmer
stemmer = ISRIStemmer()

def preprocess(text):
    text = re.sub(r'[ً-ٟؐ-ؚ]', '', text)   # tashkeel
    text = re.sub(r'ـ', '', text)                           # tatweel
    text = re.sub(r'[آأإٱ]', 'ا', text) # alef
    text = re.sub(r'[^؀-ۿ\s]', '', text)              # noise
    tokens = text.split()
    tokens = [re.sub(r'[^ء-غف-ي٠-٩]', '', t)
              for t in tokens]
    tokens = [t for t in tokens if t and t not in arabic_stopwords]
    tokens = [stemmer.stem(t) if len(t) >= 4 else t for t in tokens]
    return ' '.join(tokens)


def answer_question_retrieval(question_ar):
    q_clean  = preprocess(question_ar)
    q_flat   = get_flat_embeddings([q_clean])

    # Search across entire dataset (no category filter)
    sims     = cosine_similarity(q_flat, all_flat_emb)[0]
    best_idx = np.argmax(sims)

    return {
        'question'  : question_ar,
        'category'  : df['Category'].iloc[best_idx],
        'answer'    : df['Answer'].iloc[best_idx],
        'similarity': round(float(sims.max()), 4)
    }

In [69]:
# Test
result = answer_question_retrieval('ما هو البرلمان الأوروبي؟')
print(f"Category  : {result['category']}")
print(f"Answer    : {result['answer']}")
print(f"Similarity: {result['similarity']}")


Category  : السياسة والقانون
Answer    : كما يوجد بعض الدول التي يمكن اعتبارها إسلامية وليست عضوا في منظمة المؤتمر الإسلامي، مثل البوسنة والهرسك.
Similarity: 0.7011


---
# Part 2: Seq2Seq Model with Various Word Embedding Methods
Train an LSTM encoder-decoder (Seq2Seq) to generate answers from questions using three embedding strategies:
- **Traditional** — Keras Tokenizer + learned Embedding layer
- **Independent** — Word2Vec pretrained embeddings (frozen)
- **Dependent** — AraBERT contextual sequence embeddings


In [70]:
!pip install nltk gensim -q
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download('punkt', quiet=True)

smooth = SmoothingFunction().method1

def bleu_score(ref, hyp):
    ref_tokens = str(ref).split()
    hyp_tokens = str(hyp).split()
    if not hyp_tokens:
        return 0.0
    return sentence_bleu([ref_tokens], hyp_tokens, smoothing_function=smooth)

def mean_bleu(refs, hyps):
    scores = [bleu_score(r, h) for r, h in zip(refs, hyps)]
    return float(np.mean(scores))

print("BLEU helper ready")


BLEU helper ready ✓


### Seq2Seq Data Preparation
---

In [71]:
from sklearn.model_selection import train_test_split as tts
from tensorflow.keras.preprocessing.text import Tokenizer as KTok
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Q/A pairs — use preprocessed question + original answer
qa_df = df[['stemmed_question_text', 'Answer']].dropna().copy()
qa_df.columns = ['question', 'answer']
qa_df = qa_df[(qa_df['question'].str.strip() != '') & (qa_df['answer'].str.strip() != '')].reset_index(drop=True)

# Add start/end tokens for decoder
qa_df['dec_in']  = 'startseq ' + qa_df['answer']
qa_df['dec_tgt'] = qa_df['answer'] + ' endseq'

seq_train, seq_test = tts(qa_df, test_size=0.2, random_state=42)
seq_train = seq_train.reset_index(drop=True)
seq_test  = seq_test.reset_index(drop=True)

print(f"Seq2Seq train: {len(seq_train)} | test: {len(seq_test)}")
print(seq_train.head(2))


Seq2Seq train: 3617 | test: 905
                          question  \
0  لمذ تعد درس جدى اهم خطو عدد شرع   
1                  علم عمد نظم وقت   

                                              answer  \
0  تعد دراسة الجدوى أهم خطوات إعداد المشروع لأنها...   
1  أعتمد في تنظيم وقتي على الجداول الزمنية والأول...   

                                              dec_in  \
0  startseq تعد دراسة الجدوى أهم خطوات إعداد المش...   
1  startseq أعتمد في تنظيم وقتي على الجداول الزمن...   

                                             dec_tgt  
0  تعد دراسة الجدوى أهم خطوات إعداد المشروع لأنها...  
1  أعتمد في تنظيم وقتي على الجداول الزمنية والأول...  


### Answer Tokenizer (shared across all Seq2Seq methods)
---

In [72]:
ANS_MAX_LEN = 40

ans_tok = KTok(filters='', oov_token='<OOV>')
ans_tok.fit_on_texts(pd.concat([seq_train['dec_in'], seq_train['dec_tgt']]).astype(str))
ans_vocab_size = len(ans_tok.word_index) + 1

def encode_answers(texts, maxlen=ANS_MAX_LEN):
    seqs = ans_tok.texts_to_sequences(texts.astype(str).tolist())
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

train_dec_in  = encode_answers(seq_train['dec_in'])
train_dec_tgt = encode_answers(seq_train['dec_tgt'])
test_dec_in   = encode_answers(seq_test['dec_in'])
test_dec_tgt  = encode_answers(seq_test['dec_tgt'])

print(f"Answer vocab size : {ans_vocab_size}")
print(f"Decoder input shape: {train_dec_in.shape}")


Answer vocab size : 15791
Decoder input shape: (3617, 40)


### Seq2Seq Architecture & Evaluation Helper
---

In [73]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
import tensorflow as tf

LATENT_DIM     = 256
SEQ2SEQ_EPOCHS = 10
SEQ2SEQ_BATCH  = 64

# Separate early stopping for Seq2Seq (avoids sharing state with Part 1's es)
es_seq = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

def build_seq2seq_from_sequence(enc_input_layer, enc_input_tensor, tag=''):
    """Build Seq2Seq given encoder input.
    tag: unique string to avoid duplicate Keras layer names across calls.
    """
    _, state_h, state_c = LSTM(LATENT_DIM, return_state=True,
                                name=f'enc_lstm_{tag}')(enc_input_tensor)

    dec_input = Input(shape=(ANS_MAX_LEN,), name=f'dec_input_{tag}')
    dec_emb   = Embedding(ans_vocab_size, 128, mask_zero=True,
                           name=f'dec_emb_{tag}')(dec_input)
    dec_lstm  = LSTM(LATENT_DIM, return_sequences=True, return_state=True,
                     name=f'dec_lstm_{tag}')
    dec_out, _, _ = dec_lstm(dec_emb, initial_state=[state_h, state_c])
    dec_dense = Dense(ans_vocab_size, activation='softmax',
                      name=f'dec_dense_{tag}')(dec_out)

    train_model = Model([enc_input_layer, dec_input], dec_dense)
    train_model.compile(loss='sparse_categorical_crossentropy',
                        optimizer='adam', metrics=['accuracy'])
    return train_model

def bleu_from_teacher_forcing(model, X_enc, n=50):
    """Evaluate BLEU using teacher-forcing predictions on test set."""
    n = min(n, len(seq_test))
    preds = model.predict([X_enc[:n], test_dec_in[:n]], verbose=0)
    pred_ids = np.argmax(preds, axis=-1)

    hyps, refs = [], []
    for i in range(n):
        hyp = [ans_tok.index_word.get(t, '') for t in pred_ids[i] if t > 0]
        hyp = [w for w in hyp if w not in ('', '<OOV>', 'startseq', 'endseq')]
        ref = [ans_tok.index_word.get(t, '') for t in test_dec_tgt[i] if t > 0]
        ref = [w for w in ref if w not in ('', '<OOV>', 'startseq', 'endseq')]
        if ref:
            hyps.append(' '.join(hyp))
            refs.append(' '.join(ref))

    return mean_bleu(refs, hyps)

print("Seq2Seq helpers ready ✓")


Seq2Seq helpers ready ✓


### Method 1: TF-IDF Encoding
---

In [74]:
from sklearn.feature_extraction.text import TfidfVectorizer

MAX_FEATURES = 500

tfidf_vec = TfidfVectorizer(max_features=MAX_FEATURES)
tfidf_vec.fit(seq_train['question'].tolist())

def encode_tfidf(texts):
    # Dense array reshaped to (batch, 1, features) for LSTM encoder
    arr = tfidf_vec.transform(texts.tolist()).toarray().astype('float32')
    return arr.reshape(arr.shape[0], 1, arr.shape[1])

X_enc_tfidf_train = encode_tfidf(seq_train['question'])
X_enc_tfidf_test  = encode_tfidf(seq_test['question'])
print(f"TF-IDF encoder input shape: {X_enc_tfidf_train.shape}")


TF-IDF encoder input shape: (3617, 1, 500)


In [75]:
# Build Seq2Seq: encoder takes TF-IDF vectors (no embedding layer)
enc_inp_tfidf = Input(shape=(1, MAX_FEATURES), name='enc_input_tfidf')
seq2seq_tfidf = build_seq2seq_from_sequence(enc_inp_tfidf, enc_inp_tfidf, tag='tfidf')
seq2seq_tfidf.summary()


Model: "functional_23"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ dec_input_tfidf     │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_input_tfidf     │ (None, 1, 500)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_emb_tfidf       │ (None, 40, 128)   │  2,021,248 │ dec_input_tfidf[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_tfidf      │ [(None, 256),     │    775,168 │ enc_input_tfidf[… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_tfidf      │ [(None, 40, 256), │    394,240 │ dec_emb_tfidf[0]… │
│ (LSTM)              │ (None, 256),      │            │ enc_lstm_tfidf[0… │
│                     │ (None, 256)]      │            │ enc_lstm_tfidf[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_dense_tfidf     │ (None, 40, 15791) │  4,058,287 │ dec_lstm_tfidf[0… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,248,943 (27.65 MB)

 Trainable params: 7,248,943 (27.65 MB)

 Non-trainable params: 0 (0.00 B)

In [76]:
seq2seq_tfidf.fit(
    [X_enc_tfidf_train, train_dec_in], train_dec_tgt,
    epochs=SEQ2SEQ_EPOCHS,
    batch_size=SEQ2SEQ_BATCH,
    validation_split=0.1,
    callbacks=[es_seq]
)

bleu_tfidf = bleu_from_teacher_forcing(seq2seq_tfidf, X_enc_tfidf_test)
print(f"\nSeq2Seq (TF-IDF) BLEU: {bleu_tfidf:.4f}")


Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 13s 202ms/step - accuracy: 0.0538 - loss: 8.9519 - val_accuracy: 0.0569 - val_loss: 8.5544
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - accuracy: 0.0557 - loss: 8.1384 - val_accuracy: 0.0569 - val_loss: 8.4152
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 144ms/step - accuracy: 0.0591 - loss: 7.8730 - val_accuracy: 0.0649 - val_loss: 8.3350
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 133ms/step - accuracy: 0.0685 - loss: 7.6657 - val_accuracy: 0.0720 - val_loss: 8.3035
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 10s 127ms/step - accuracy: 0.0725 - loss: 7.4945 - val_accuracy: 0.0747 - val_loss: 8.2912
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 142ms/step - accuracy: 0.0765 - loss: 7.3613 - val_accuracy: 0.0765 - val_loss: 8.3242
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 135ms/step - accuracy: 0.0805 - loss: 7.2511 - val_accuracy: 0.0804 - val_loss: 8.3464
Epoch 8/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 7s 140ms/step - accuracy: 0.0834 - loss: 7.1425 - val_accuracy: 

### Method 2: N-gram Encoding
---

In [77]:
from sklearn.feature_extraction.text import CountVectorizer

ngram_vec = CountVectorizer(ngram_range=(1, 2), max_features=MAX_FEATURES)
ngram_vec.fit(seq_train['question'].tolist())

def encode_ngram(texts):
    arr = ngram_vec.transform(texts.tolist()).toarray().astype('float32')
    return arr.reshape(arr.shape[0], 1, arr.shape[1])

X_enc_ngram_train = encode_ngram(seq_train['question'])
X_enc_ngram_test  = encode_ngram(seq_test['question'])
print(f"N-gram encoder input shape: {X_enc_ngram_train.shape}")


N-gram encoder input shape: (3617, 1, 500)


In [78]:
# Build Seq2Seq: encoder takes N-gram vectors (no embedding layer)
enc_inp_ngram = Input(shape=(1, MAX_FEATURES), name='enc_input_ngram')
seq2seq_ngram = build_seq2seq_from_sequence(enc_inp_ngram, enc_inp_ngram, tag='ngram')
seq2seq_ngram.summary()


Model: "functional_24"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ dec_input_ngram     │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_input_ngram     │ (None, 1, 500)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_emb_ngram       │ (None, 40, 128)   │  2,021,248 │ dec_input_ngram[… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_ngram      │ [(None, 256),     │    775,168 │ enc_input_ngram[… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_ngram      │ [(None, 40, 256), │    394,240 │ dec_emb_ngram[0]… │
│ (LSTM)              │ (None, 256),      │            │ enc_lstm_ngram[0… │
│                     │ (None, 256)]      │            │ enc_lstm_ngram[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_dense_ngram     │ (None, 40, 15791) │  4,058,287 │ dec_lstm_ngram[0… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,248,943 (27.65 MB)

 Trainable params: 7,248,943 (27.65 MB)

 Non-trainable params: 0 (0.00 B)

In [79]:
seq2seq_ngram.fit(
    [X_enc_ngram_train, train_dec_in], train_dec_tgt,
    epochs=SEQ2SEQ_EPOCHS,
    batch_size=SEQ2SEQ_BATCH,
    validation_split=0.1,
    callbacks=[es_seq]
)

bleu_ngram = bleu_from_teacher_forcing(seq2seq_ngram, X_enc_ngram_test)
print(f"\nSeq2Seq (N-gram) BLEU: {bleu_ngram:.4f}")


Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 14s 162ms/step - accuracy: 0.0539 - loss: 8.9549 - val_accuracy: 0.0569 - val_loss: 8.5641
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 9s 138ms/step - accuracy: 0.0555 - loss: 8.1425 - val_accuracy: 0.0574 - val_loss: 8.4089
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 122ms/step - accuracy: 0.0591 - loss: 7.8683 - val_accuracy: 0.0649 - val_loss: 8.3312

Seq2Seq (N-gram) BLEU: 0.0001


### Method 3: Independent Embedding (Word2Vec — frozen pretrained)
---

In [80]:
from gensim.models import Word2Vec as GW2V
from tensorflow.keras.preprocessing.text import Tokenizer as KTok2
from tensorflow.keras.preprocessing.sequence import pad_sequences

W2V_DIM   = 100
Q_MAX_LEN = 50

q_tok_w2v = KTok2(num_words=20000, oov_token='<OOV>')
q_tok_w2v.fit_on_texts(seq_train['question'].tolist())

def encode_q_seq(texts, tok, maxlen):
    seqs = tok.texts_to_sequences(texts.tolist())
    return pad_sequences(seqs, maxlen=maxlen, padding='post', truncating='post')

X_enc_w2v_train_seq = encode_q_seq(seq_train['question'], q_tok_w2v, Q_MAX_LEN)
X_enc_w2v_test_seq  = encode_q_seq(seq_test['question'],  q_tok_w2v, Q_MAX_LEN)

# Train Word2Vec
w2v_sentences = [t.split() for t in seq_train['question'].tolist()]
w2v_model_seq = GW2V(w2v_sentences, vector_size=W2V_DIM, window=5, min_count=1, workers=4)

q_vocab_size_w2v = len(q_tok_w2v.word_index) + 1
emb_matrix_seq = np.zeros((q_vocab_size_w2v, W2V_DIM))
for word, idx in q_tok_w2v.word_index.items():
    if word in w2v_model_seq.wv:
        emb_matrix_seq[idx] = w2v_model_seq.wv[word]

print(f"Word2Vec encoder shape: {X_enc_w2v_train_seq.shape} | matrix: {emb_matrix_seq.shape}")


Word2Vec encoder shape: (3617, 50) | matrix: (2272, 100)


In [81]:
enc_inp_w2v = Input(shape=(Q_MAX_LEN,), name='enc_input_w2v')
enc_emb_w2v = Embedding(q_vocab_size_w2v, W2V_DIM,
                         weights=[emb_matrix_seq],
                         trainable=False,
                         name='enc_emb_w2v')(enc_inp_w2v)

seq2seq_w2v = build_seq2seq_from_sequence(enc_inp_w2v, enc_emb_w2v, tag='w2v')
seq2seq_w2v.summary()


Model: "functional_25"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ enc_input_w2v       │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_input_w2v       │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_emb_w2v         │ (None, 50, 100)   │    227,200 │ enc_input_w2v[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_emb_w2v         │ (None, 40, 128)   │  2,021,248 │ dec_input_w2v[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_w2v (LSTM) │ [(None, 256),     │    365,568 │ enc_emb_w2v[0][0] │
│                     │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_w2v (LSTM) │ [(None, 40, 256), │    394,240 │ dec_emb_w2v[0][0… │
│                     │ (None, 256),      │            │ enc_lstm_w2v[0][… │
│                     │ (None, 256)]      │            │ enc_lstm_w2v[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_dense_w2v       │ (None, 40, 15791) │  4,058,287 │ dec_lstm_w2v[0][… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,066,543 (26.96 MB)

 Trainable params: 6,839,343 (26.09 MB)

 Non-trainable params: 227,200 (887.50 KB)

In [82]:
seq2seq_w2v.fit(
    [X_enc_w2v_train_seq, train_dec_in], train_dec_tgt,
    epochs=SEQ2SEQ_EPOCHS,
    batch_size=SEQ2SEQ_BATCH,
    validation_split=0.1,
    callbacks=[es_seq]
)

bleu_w2v = bleu_from_teacher_forcing(seq2seq_w2v, X_enc_w2v_test_seq)
print(f"\nSeq2Seq (Word2Vec / Independent) BLEU: {bleu_w2v:.4f}")


Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 11s 122ms/step - accuracy: 0.0542 - loss: 8.8303 - val_accuracy: 0.0567 - val_loss: 8.4485
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 111ms/step - accuracy: 0.0645 - loss: 8.0552 - val_accuracy: 0.0681 - val_loss: 8.3198
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.0710 - loss: 7.7741 - val_accuracy: 0.0722 - val_loss: 8.2408
Epoch 4/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.0717 - loss: 7.5746 - val_accuracy: 0.0717 - val_loss: 8.2318
Epoch 5/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 110ms/step - accuracy: 0.0737 - loss: 7.4208 - val_accuracy: 0.0769 - val_loss: 8.2425
Epoch 6/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.0787 - loss: 7.2897 - val_accuracy: 0.0817 - val_loss: 8.2634
Epoch 7/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 108ms/step - accuracy: 0.0818 - loss: 7.1649 - val_accuracy: 0.0802 - val_loss: 8.2740



Seq2Seq (Word2Vec / Independent) BLEU: 0.0071


### Method 4: Dependent Embedding (AraBERT — contextual sequence embeddings)
---

In [86]:
BERT_TOKENIZER_MAX_LEN = 30 # Define this explicitly to ensure consistent padding

def get_bert_seq_embeddings(texts, batch_size=32):
    all_embs = []
    texts_list = texts.tolist()
    for i in range(0, len(texts_list), batch_size):
        batch = texts_list[i:i+batch_size]
        inputs = tokenizer_bert(batch, return_tensors='pt', padding=True,
                                truncation=True, max_length=BERT_TOKENIZER_MAX_LEN).to(DEVICE)
        with torch.no_grad():
            out = model_bert(**inputs)
        all_embs.append(out.last_hidden_state.cpu().numpy())

    # Ensure all embeddings are padded to the consistent BERT_TOKENIZER_MAX_LEN
    padded_embs = []
    for e in all_embs:
        current_seq_len = e.shape[1]
        if current_seq_len < BERT_TOKENIZER_MAX_LEN:
            pad_amount = BERT_TOKENIZER_MAX_LEN - current_seq_len
            padded_e = np.pad(e, ((0,0), (0, pad_amount), (0,0)), mode='constant')
            padded_embs.append(padded_e)
        elif current_seq_len > BERT_TOKENIZER_MAX_LEN:
            # This case should ideally not happen if tokenizer truncation works as expected
            truncated_e = e[:, :BERT_TOKENIZER_MAX_LEN, :]
            padded_embs.append(truncated_e)
        else:
            padded_embs.append(e)

    return np.vstack(padded_embs)

X_enc_bert_train = get_bert_seq_embeddings(seq_train['question'])
X_enc_bert_test  = get_bert_seq_embeddings(seq_test['question'])
BERT_SEQ_LEN = BERT_TOKENIZER_MAX_LEN # Now BERT_SEQ_LEN is fixed to the tokenizer's max_length
print(f"AraBERT encoder input shape: {X_enc_bert_train.shape}")

AraBERT encoder input shape: (3617, 30, 768)


In [87]:
enc_inp_bert = Input(shape=(BERT_SEQ_LEN, 768), name='enc_input_bert')
seq2seq_bert = build_seq2seq_from_sequence(enc_inp_bert, enc_inp_bert, tag='bert')
seq2seq_bert.summary()


Model: "functional_27"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ dec_input_bert      │ (None, 40)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_input_bert      │ (None, 30, 768)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_emb_bert        │ (None, 40, 128)   │  2,021,248 │ dec_input_bert[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ enc_lstm_bert       │ [(None, 256),     │  1,049,600 │ enc_input_bert[0… │
│ (LSTM)              │ (None, 256),      │            │                   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_lstm_bert       │ [(None, 40, 256), │    394,240 │ dec_emb_bert[0][… │
│ (LSTM)              │ (None, 256),      │            │ enc_lstm_bert[0]… │
│                     │ (None, 256)]      │            │ enc_lstm_bert[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dec_dense_bert      │ (None, 40, 15791) │  4,058,287 │ dec_lstm_bert[0]… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 7,523,375 (28.70 MB)

 Trainable params: 7,523,375 (28.70 MB)

 Non-trainable params: 0 (0.00 B)

In [88]:
seq2seq_bert.fit(
    [X_enc_bert_train, train_dec_in], train_dec_tgt,
    epochs=SEQ2SEQ_EPOCHS,
    batch_size=SEQ2SEQ_BATCH,
    validation_split=0.1,
    callbacks=[es_seq]
)

bleu_bert = bleu_from_teacher_forcing(seq2seq_bert, X_enc_bert_test)
print(f"\nSeq2Seq (AraBERT / Dependent) BLEU: {bleu_bert:.4f}")


Epoch 1/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 8s 125ms/step - accuracy: 0.0507 - loss: 8.7430 - val_accuracy: 0.0567 - val_loss: 8.4114
Epoch 2/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 113ms/step - accuracy: 0.0680 - loss: 8.0432 - val_accuracy: 0.0701 - val_loss: 8.3307
Epoch 3/10
51/51 ━━━━━━━━━━━━━━━━━━━━ 6s 116ms/step - accuracy: 0.0705 - loss: 7.7861 - val_accuracy: 0.0731 - val_loss: 8.2704

Seq2Seq (AraBERT / Dependent) BLEU: 0.0002


### Seq2Seq Results Comparison
---

In [89]:
seq2seq_results = pd.DataFrame([
    {'Embedding Method': 'TF-IDF',                                       'Type': 'Traditional', 'BLEU': round(bleu_tfidf, 4)},
    {'Embedding Method': 'N-gram (unigram + bigram)',                     'Type': 'Traditional', 'BLEU': round(bleu_ngram, 4)},
    {'Embedding Method': 'Word2Vec (frozen pretrained)',                  'Type': 'Independent', 'BLEU': round(bleu_w2v,   4)},
    {'Embedding Method': 'AraBERT (contextual sequence embeddings)',      'Type': 'Dependent',   'BLEU': round(bleu_bert,  4)},
]).sort_values('BLEU', ascending=False).reset_index(drop=True)

print("Seq2Seq BLEU Comparison:")
display(seq2seq_results)
best_seq2seq = seq2seq_results.iloc[0]['Embedding Method']
print(f"\nBest Seq2Seq method: {best_seq2seq}")


Seq2Seq BLEU Comparison:


,Embedding Method,Type,BLEU
0,TF-IDF,Traditional,0.0082
1,Word2Vec (frozen pretrained),Independent,0.0071
2,AraBERT (contextual sequence embeddings),Dependent,0.0002
3,N-gram (unigram + bigram),Traditional,0.0001



Best Seq2Seq method: TF-IDF


---
# Part 3: Transformer-Based Models — Fine-tuned Locally (No API)
Fine-tune three transformer models on the AAFAQ Q/A pairs:
- **AraGPT2** (GPT-family)
- **mT5-small** (T5-family)
- **Qwen2.5-0.5B-Instruct** (Qwen-family)

All models are loaded locally and fine-tuned using HuggingFace Trainer. No API used.


In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate peft bitsandbytes

import gc, os, torch, tensorflow as tf
from datasets import Dataset as HFDataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq, DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model
import evaluate

# ── Free GPU from TF so PyTorch gets all VRAM ──────────────────────────
tf.keras.backend.clear_session()
tf.config.set_visible_devices([], 'GPU')
for _v in ['model_bert']:
    if _v in globals(): del globals()[_v]
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Device: {DEVICE} | Free VRAM: {free:.2f} GB")


In [ ]:
# Use the same Q/A pairs from Seq2Seq section
TRAIN_LIMIT = min(1500, len(seq_train))
EVAL_LIMIT  = min(200,  len(seq_test))

ft_qs_train = seq_train['question'][:TRAIN_LIMIT].tolist()
ft_as_train = seq_train['answer'][:TRAIN_LIMIT].tolist()
ft_qs_eval  = seq_test['question'][:EVAL_LIMIT].tolist()
ft_as_eval  = seq_test['answer'][:EVAL_LIMIT].tolist()

print(f"Fine-tuning: {TRAIN_LIMIT} train | {EVAL_LIMIT} eval samples")


In [ ]:
import evaluate as hf_evaluate

sacrebleu = hf_evaluate.load("sacrebleu")

def compute_bleu(generate_fn, questions, references, n=50):
    preds = []
    for q in questions[:n]:
        try:
            preds.append(generate_fn(str(q)))
        except Exception:
            preds.append('')
    refs = [[r] for r in references[:n]]
    result = sacrebleu.compute(predictions=preds, references=refs)
    return round(result['score'], 4), preds

print("BLEU helper ready ✓")


---
## 3.1 mT5-small (T5) — Fine-tuned with Seq2SeqTrainer
---

In [ ]:
MT5_NAME = 'google/mt5-small'
MT5_OUT  = '/content/mt5_finetuned'

mt5_tokenizer = AutoTokenizer.from_pretrained(MT5_NAME)

def preprocess_mt5(questions, answers):
    inputs = mt5_tokenizer(
        ["السؤال: " + q for q in questions],
        truncation=True, padding='max_length', max_length=128
    )
    labels = mt5_tokenizer(
        text_target=answers, truncation=True, padding='max_length', max_length=64
    )
    inputs["labels"] = labels["input_ids"]
    return inputs

mt5_train_dict = preprocess_mt5(ft_qs_train, ft_as_train)
mt5_eval_dict  = preprocess_mt5(ft_qs_eval,  ft_as_eval)

mt5_train_ds = HFDataset.from_dict(mt5_train_dict)
mt5_eval_ds  = HFDataset.from_dict(mt5_eval_dict)

print(f"mT5 dataset: {len(mt5_train_ds)} train | {len(mt5_eval_ds)} eval")


In [ ]:
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_NAME)
mt5_model.to(DEVICE)

data_collator_mt5 = DataCollatorForSeq2Seq(tokenizer=mt5_tokenizer, model=mt5_model)

mt5_args = Seq2SeqTrainingArguments(
    output_dir=MT5_OUT,
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    predict_with_generate=True,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
)

mt5_trainer = Seq2SeqTrainer(
    model=mt5_model,
    args=mt5_args,
    train_dataset=mt5_train_ds,
    eval_dataset=mt5_eval_ds,
    data_collator=data_collator_mt5,
)

print("Fine-tuning mT5-small (T5)...")
mt5_trainer.train()
print("mT5 fine-tuning complete ✓")


In [ ]:
def generate_mt5(question, max_new=80):
    inputs = mt5_tokenizer(
        "السؤال: " + question,
        return_tensors='pt', truncation=True, max_length=128
    ).to(DEVICE)
    with torch.no_grad():
        out = mt5_model.generate(**inputs, max_new_tokens=max_new, num_beams=4)
    return mt5_tokenizer.decode(out[0], skip_special_tokens=True)

mt5_bleu, mt5_preds = compute_bleu(generate_mt5, ft_qs_eval, ft_as_eval)
print(f"mT5-small BLEU: {mt5_bleu}")

# Free GPU before loading next model
del mt5_model
gc.collect(); torch.cuda.empty_cache()
print("GPU freed after mT5 ✓")


---
## 3.2 AraGPT2 (GPT) — Fine-tuned with Trainer
---

In [ ]:
ARAGPT_NAME = 'aubmindlab/aragpt2-base'
ARAGPT_OUT  = '/content/aragpt_finetuned'

aragpt_tokenizer = AutoTokenizer.from_pretrained(ARAGPT_NAME)
aragpt_tokenizer.pad_token = aragpt_tokenizer.eos_token

def preprocess_aragpt(questions, answers):
    texts = [f"السؤال: {q} الإجابة: {a}" for q, a in zip(questions, answers)]
    return aragpt_tokenizer(texts, truncation=True, padding='max_length', max_length=128)

aragpt_train_ds = HFDataset.from_dict(preprocess_aragpt(ft_qs_train, ft_as_train))
aragpt_eval_ds  = HFDataset.from_dict(preprocess_aragpt(ft_qs_eval,  ft_as_eval))

aragpt_train_ds = aragpt_train_ds.remove_columns(
    [c for c in aragpt_train_ds.column_names if c not in ['input_ids','attention_mask']]
)
aragpt_eval_ds  = aragpt_eval_ds.remove_columns(
    [c for c in aragpt_eval_ds.column_names if c not in ['input_ids','attention_mask']]
)

print(f"AraGPT2 dataset: {len(aragpt_train_ds)} train | {len(aragpt_eval_ds)} eval")


In [ ]:
aragpt_model = AutoModelForCausalLM.from_pretrained(ARAGPT_NAME)
aragpt_model.to(DEVICE)

# DataCollatorForLanguageModeling handles label masking automatically (mlm=False for causal LM)
data_collator_gpt = DataCollatorForLanguageModeling(tokenizer=aragpt_tokenizer, mlm=False)

aragpt_args = TrainingArguments(
    output_dir=ARAGPT_OUT,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='no',
    fp16=(DEVICE == 'cuda'),
    report_to='none',
)

aragpt_trainer = Trainer(
    model=aragpt_model,
    args=aragpt_args,
    train_dataset=aragpt_train_ds,
    eval_dataset=aragpt_eval_ds,
    data_collator=data_collator_gpt,
)

print("Fine-tuning AraGPT2 (GPT)...")
aragpt_trainer.train()
print("AraGPT2 fine-tuning complete ✓")


In [ ]:
def generate_aragpt(question, max_new=80):
    prompt = f"السؤال: {question} الإجابة:"
    inputs = aragpt_tokenizer(prompt, return_tensors='pt').to(DEVICE)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = aragpt_model.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=True, top_p=0.9, temperature=0.8,
            no_repeat_ngram_size=3, repetition_penalty=1.5,
            pad_token_id=aragpt_tokenizer.eos_token_id
        )
    return aragpt_tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

aragpt_bleu, aragpt_preds = compute_bleu(generate_aragpt, ft_qs_eval, ft_as_eval)
print(f"AraGPT2 BLEU: {aragpt_bleu}")

# Free GPU before loading Qwen
del aragpt_model
gc.collect(); torch.cuda.empty_cache()
print("GPU freed after AraGPT2 ✓")


---
## 3.3 Qwen2.5-0.5B-Instruct (Qwen) — Fine-tuned with LoRA
---

In [ ]:
QWEN_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
QWEN_OUT  = '/content/qwen_finetuned'

qwen_tokenizer = AutoTokenizer.from_pretrained(QWEN_NAME, trust_remote_code=True)
qwen_tokenizer.pad_token = qwen_tokenizer.eos_token

def preprocess_qwen(questions, answers):
    texts = [f"السؤال: {q} الإجابة: {a}" for q, a in zip(questions, answers)]
    return qwen_tokenizer(texts, truncation=True, max_length=128)

qwen_train_ds = HFDataset.from_dict(preprocess_qwen(ft_qs_train, ft_as_train))
qwen_eval_ds  = HFDataset.from_dict(preprocess_qwen(ft_qs_eval,  ft_as_eval))

qwen_train_ds = qwen_train_ds.remove_columns(
    [c for c in qwen_train_ds.column_names if c not in ['input_ids','attention_mask']]
)
qwen_eval_ds  = qwen_eval_ds.remove_columns(
    [c for c in qwen_eval_ds.column_names if c not in ['input_ids','attention_mask']]
)

print(f"Qwen dataset: {len(qwen_train_ds)} train | {len(qwen_eval_ds)} eval")


In [ ]:
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_NAME, trust_remote_code=True, device_map='auto'
)

# LoRA — same as professor's lab (MT_Lab_sol)
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    task_type="CAUSAL_LM"
)
qwen_model = get_peft_model(qwen_model, lora_config)
qwen_model.print_trainable_parameters()

data_collator_qwen = DataCollatorForLanguageModeling(tokenizer=qwen_tokenizer, mlm=False)

qwen_args = TrainingArguments(
    output_dir=QWEN_OUT,
    learning_rate=2e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
    gradient_accumulation_steps=4,
)

qwen_trainer = Trainer(
    model=qwen_model,
    args=qwen_args,
    train_dataset=qwen_train_ds,
    eval_dataset=qwen_eval_ds,
    data_collator=data_collator_qwen,
)

print("Fine-tuning Qwen2.5-0.5B-Instruct...")
qwen_trainer.train()
print("Qwen fine-tuning complete ✓")


In [ ]:
def generate_qwen(question, max_new=80):
    prompt = f"السؤال: {question} الإجابة:"
    inputs = qwen_tokenizer(prompt, return_tensors='pt').to(qwen_model.device)
    input_len = inputs['input_ids'].shape[1]
    with torch.no_grad():
        out = qwen_model.generate(
            **inputs, max_new_tokens=max_new,
            do_sample=False, no_repeat_ngram_size=2,
            pad_token_id=qwen_tokenizer.eos_token_id
        )
    return qwen_tokenizer.decode(out[0][input_len:], skip_special_tokens=True).strip()

qwen_bleu, qwen_preds = compute_bleu(generate_qwen, ft_qs_eval, ft_as_eval)
print(f"Qwen2.5-0.5B BLEU: {qwen_bleu}")


---
## Final Results: All Approaches Compared
---

In [ ]:
import pandas as pd

transformer_results = pd.DataFrame([
    {'Model': 'mT5-small (T5)',            'Type': 'Seq2Seq Transformer',  'BLEU': mt5_bleu},
    {'Model': 'AraGPT2 (GPT)',             'Type': 'Causal Transformer',   'BLEU': aragpt_bleu},
    {'Model': 'Qwen2.5-0.5B-Instruct',     'Type': 'Causal Transformer',   'BLEU': qwen_bleu},
])

seq2seq_final = seq2seq_results.rename(columns={'Embedding Method': 'Model'}).copy()
seq2seq_final['Type'] = 'LSTM Seq2Seq'

final_results = pd.concat([seq2seq_final, transformer_results], ignore_index=True)                   .sort_values('BLEU', ascending=False).reset_index(drop=True)

print("=" * 55)
print("FINAL BLEU COMPARISON — ALL METHODS")
print("=" * 55)
display(final_results)
print(f"\nBest: {final_results.iloc[0]['Model']} (BLEU = {final_results.iloc[0]['BLEU']})")


In [ ]:
# Sample outputs
print("\nSample Generated Answers (5 questions):")
print("=" * 70)
for i in range(min(5, len(ft_qs_eval))):
    print(f"Q: {ft_qs_eval[i]}")
    print(f"  Reference : {ft_as_eval[i][:100]}")
    print(f"  mT5       : {mt5_preds[i][:100]}")
    print(f"  AraGPT2   : {aragpt_preds[i][:100]}")
    print(f"  Qwen      : {qwen_preds[i][:100]}")
    print()
